# 01 - Exploratory Data Analysis (EDA)

**Goal**: Look at our data before building any model.

EDA = Data's physical exam. Just like you wouldn't diagnose a patient without checking vitals first.

**What we'll check:**
1. Basic stats (how many students, score ranges)
2. Distribution of each block score (are they normal?)
3. Correlation heatmap (which blocks predict CBSE?)
4. Scatter plots (block scores vs CBSE score)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data_loader import load_and_merge, get_feature_columns

# Style settings for clean, portfolio-ready plots
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Load & Preview Data

In [ ]:
# Load merged data (block scores + CBSE results)
df = load_and_merge()
feature_cols = get_feature_columns(df)

print(f"\nFeature columns (block scores): {feature_cols}")
print(f"Target: cbse_score")
df.head(10)

In [ ]:
# Basic statistics
# .describe() gives count, mean, std, min, 25%, 50%, 75%, max
df[feature_cols + ['cbse_score']].describe().round(1)

## 2. Distribution of Block Scores

**Why**: Check if scores look "normal" (bell-shaped). Outliers or weird distributions = data quality issue.

**PM note**: You should be able to glance at these and say "looks reasonable" or "something is off".

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Distribution of Block Scores and CBSE Score', fontsize=16, fontweight='bold')

all_cols = feature_cols + ['cbse_score']
for i, col in enumerate(all_cols):
    ax = axes[i // 4][i % 4]
    sns.histplot(df[col], bins=25, kde=True, ax=ax, color='steelblue')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    # Add mean line
    ax.axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean={df[col].mean():.1f}')
    ax.legend(fontsize=8)

# Hide empty subplot(s)
for j in range(len(all_cols), 12):
    axes[j // 4][j % 4].set_visible(False)

plt.tight_layout()
plt.savefig('../output/distributions.png', bbox_inches='tight')
plt.show()
print('Saved to output/distributions.png')

## 3. Correlation Heatmap

**The most important chart in EDA.**

Read the CBSE row/column — it tells you which blocks are the strongest predictors.

- r > 0.7: Strong correlation
- r = 0.4-0.7: Moderate
- r < 0.4: Weak
- Look for feature-feature correlations too (potential redundancy)

In [ ]:
# Compute correlation matrix
corr_matrix = df[feature_cols + ['cbse_score']].corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,          # Show r values in each cell
    fmt='.2f',           # 2 decimal places
    cmap='RdBu_r',       # Red = positive, Blue = negative
    center=0,            # White = 0 correlation
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Correlation Heatmap: Block Scores vs CBSE Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved to output/correlation_heatmap.png')

In [ ]:
# Extract and rank correlations with CBSE score specifically
cbse_corr = corr_matrix['cbse_score'].drop('cbse_score').sort_values(ascending=False)
print('=== Correlation with CBSE Score (ranked) ===')
for block, r in cbse_corr.items():
    strength = 'STRONG' if abs(r) > 0.7 else 'moderate' if abs(r) > 0.4 else 'weak'
    print(f'  {block:25s}  r = {r:.3f}  ({strength})')

## 4. Scatter Plots: Block Scores vs CBSE Score

**Why**: Correlation tells you the strength, scatter plot tells you the shape.

Look for:
- Linear trend → good, linear regression can handle it
- Curved trend → may need non-linear model
- Outliers → data quality issue or special cases

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Block Scores vs CBSE Score', fontsize=16, fontweight='bold')

for i, col in enumerate(feature_cols):
    ax = axes[i // 5][i % 5]
    ax.scatter(df[col], df['cbse_score'], alpha=0.3, s=15, color='steelblue')
    
    # Add trend line
    z = np.polyfit(df[col], df['cbse_score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2, alpha=0.7)
    
    r = df[col].corr(df['cbse_score'])
    ax.set_title(f'{col}\nr = {r:.2f}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('CBSE Score')

plt.tight_layout()
plt.savefig('../output/scatter_plots.png', bbox_inches='tight')
plt.show()
print('Saved to output/scatter_plots.png')

## 5. Missing Values Check

**Why**: Missing data is the #1 data quality issue in real-world projects.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('=== Missing Values Report ===')
print(missing_report)

if missing.sum() == 0:
    print('\nNo missing values found. (Expected for synthetic data, rare for real data!)')

## 6. CBSE Score Distribution — Pass/Fail Analysis

**Why**: We need to understand how many students are near the passing threshold.

In [ ]:
PASS_THRESHOLD = 194  # USMLE Step 1 old passing score reference

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(df['cbse_score'], bins=30, kde=True, color='steelblue', ax=ax)
ax.axvline(PASS_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Pass threshold = {PASS_THRESHOLD}')
ax.axvline(df['cbse_score'].mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean = {df["cbse_score"].mean():.0f}')

n_fail = (df['cbse_score'] < PASS_THRESHOLD).sum()
n_total = len(df)
fail_pct = n_fail / n_total * 100

ax.set_title(f'CBSE Score Distribution\n{n_fail}/{n_total} students ({fail_pct:.1f}%) below passing threshold',
             fontsize=14, fontweight='bold')
ax.set_xlabel('CBSE Score')
ax.set_ylabel('Count')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('../output/cbse_distribution.png', bbox_inches='tight')
plt.show()
print(f'Saved to output/cbse_distribution.png')

## EDA Summary

After running all cells above, you should be able to answer:

1. **Are the score distributions reasonable?** (bell-shaped, no weird spikes)
2. **Which blocks correlate most strongly with CBSE?** (look at the heatmap)
3. **Are the relationships linear?** (look at scatter plots)
4. **Any missing data?** (check missing values report)
5. **How many students are at risk of failing?** (check pass/fail analysis)

Next step: Preprocessing & Feature Engineering (02_model_training.ipynb)